# Notebook 1 — Electricity Dataset Preparation

**Purpose:** Load, clean, and structure the raw Electricity dataset (OpenML ID: 151)
into a format suitable for streaming simulation in subsequent notebooks.

**Output:** `electricity_clean_v2.xlsx` — 45,000 rows × 10 columns,
structured as 90 equal batches of 500 rows each with a `batch_id` column.

| Step | Description |
|------|-------------|
| 1 | Mount Google Drive and set file paths |
| 2 | Load raw CSV from OpenML |
| 3 | Inspect dataset shape, types, and distributions |
| 4 | Clean byte-string encoding artifacts in `day` and `class` columns |
| 5 | Encode target variable (`class`: DOWN=0, UP=1, stored as integer in `class` column) |
| 6 | Trim to 45,000 rows for exact batch alignment |
| 7 | Assign `batch_id` (0-indexed, 500 rows per batch) |
| 8 | Validate and export cleaned dataset |

## Step 1 — Mount Google Drive & Configure Paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ── PATH CONFIGURATION ──────────────────────────────────────────
# Update BASE_PATH if your Drive folder structure differs.
BASE_PATH = (
    "/content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset"
)

RAW_FILE    = f"{BASE_PATH}/electricity.csv"
OUTPUT_FILE = f"{BASE_PATH}/ready_to_use_datasets/electricity_clean.xlsx"

BATCH_SIZE    = 500      # rows per streaming batch
TOTAL_SAMPLES = 45_000   # trimmed total: 90 batches × 500 rows
                         # matches all synthetic datasets exactly

print(f'Raw input  : {RAW_FILE}')
print(f'Output     : {OUTPUT_FILE}')
print(f'Batch size : {BATCH_SIZE} rows')
print(f'Total rows : {TOTAL_SAMPLES} ({TOTAL_SAMPLES // BATCH_SIZE} batches)')

Raw input  : /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/electricity.csv
Output     : /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/electricity_clean.xlsx
Batch size : 500 rows
Total rows : 45000 (90 batches)


## Step 2 — Load Raw Dataset

The raw Electricity dataset from OpenML contains 45,312 rows.
The last 312 rows form an incomplete batch (312 < 500) and will be
trimmed in Step 6 to ensure all batches are equal-sized.

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv(RAW_FILE)
print(f'Raw shape: {df.shape}')
df.head()

Raw shape: (45312, 9)


,date,day,period,nswprice,nswdemand,vicprice,vicdemand,transfer,class
0,0.0,b'2',0.000000,0.056443,0.439155,0.003467,0.422915,0.414912,b'UP'
1,0.0,b'2',0.021277,0.051699,0.415055,0.003467,0.422915,0.414912,b'UP'
2,0.0,b'2',0.042553,0.051489,0.385004,0.003467,0.422915,0.414912,b'UP'
3,0.0,b'2',0.063830,0.045485,0.314639,0.003467,0.422915,0.414912,b'UP'
4,0.0,b'2',0.085106,0.042482,0.251116,0.003467,0.422915,0.414912,b'DOWN'


## Step 3 — Dataset Inspection

Verify column types and check for missing values before cleaning.
Note that `day` and `class` are `object` dtype due to byte-string
encoding from the OpenML CSV format (e.g., `b'2'`, `b'UP'`).

In [4]:
print('--- Shape ---')
print(df.shape)
print()
print('--- Data Types ---')
print(df.dtypes)
print()
print('--- Missing Values ---')
print(df.isnull().sum())
print()
print('--- Class Distribution (raw) ---')
print(df['class'].value_counts())

--- Shape ---
(45312, 9)

--- Data Types ---
date         float64
day           object
period       float64
nswprice     float64
nswdemand    float64
vicprice     float64
vicdemand    float64
transfer     float64
class         object
dtype: object

--- Missing Values ---
date         0
day          0
period       0
nswprice     0
nswdemand    0
vicprice     0
vicdemand    0
transfer     0
class        0
dtype: int64

--- Class Distribution (raw) ---
class
b'DOWN'    26075
b'UP'      19237
Name: count, dtype: int64


## Step 4 — Clean Byte-String Encoding Artifacts

The OpenML CSV encodes categorical columns as Python byte literals
(e.g., `b'2'`, `b'UP'`). These are stripped here before encoding.
The cleaned `day` values are stored in a new column `day_clean` (integer)
and the cleaned `class` values overwrite the original `class` column.

In [5]:
# Clean 'day' column: b'2' → 2 (integer)
df['day_clean'] = (
    df['day']
    .astype(str)
    .str.replace("b'", '', regex=False)
    .str.replace("'", '', regex=False)
    .astype(int)
)

# Clean 'class' column: b'UP' → 'UP'
df['class'] = (
    df['class']
    .astype(str)
    .str.replace("b'", '', regex=False)
    .str.replace("'", '', regex=False)
)

print('day_clean unique values:', sorted(df['day_clean'].unique()))
print('class unique values    :', sorted(df['class'].unique()))

day_clean unique values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
class unique values    : ['DOWN', 'UP']


## Step 5 — Target Variable Encoding

The binary target `class` is encoded as an integer in-place:
`DOWN` → 0 (price falls), `UP` → 1 (price rises).
The column retains the name `class` throughout all subsequent notebooks,
consistent with the column naming used in the simulation engine.

In [6]:
df['class'] = df['class'].map({'DOWN': 0, 'UP': 1})

print('Target encoding:')
print(df['class'].value_counts())
print(f'Class balance: {df["class"].mean():.3f} (fraction UP)')

# Confirm no NaN from unmapped values
assert df['class'].isnull().sum() == 0, 'Unmapped class values found!'
print('\u2713 No unmapped values.')

Target encoding:
class
0    26075
1    19237
Name: count, dtype: int64
Class balance: 0.425 (fraction UP)
✓ No unmapped values.


## Step 6 — Trim to 45,000 Rows

The raw dataset has 45,312 rows. The last 312 rows form an incomplete
final batch (312 < 500 rows) and are removed so that the dataset divides
exactly into **90 complete batches of 500 rows**, matching the row count
of all synthetic datasets used in this study.

In [7]:
df = df.iloc[:TOTAL_SAMPLES].reset_index(drop=True)

print(f'Rows after trim : {len(df):,}  ({len(df) // BATCH_SIZE} complete batches)')
print(f'Rows removed    : {45312 - len(df)} (incomplete last batch)')
assert len(df) % BATCH_SIZE == 0, 'Row count not divisible by BATCH_SIZE!'
assert len(df) == TOTAL_SAMPLES,  'Unexpected row count after trim!'
print('✓ Row count validated.')

Rows after trim : 45,000  (90 complete batches)
Rows removed    : 312 (incomplete last batch)
✓ Row count validated.


## Step 7 — Assign batch_id

`batch_id` is a 0-indexed integer identifying which 500-row batch
each row belongs to. This column is used by all downstream notebooks
to simulate the streaming evaluation loop.

In [8]:
df['batch_id'] = df.index // BATCH_SIZE

# Verify all batches have exactly 500 rows
batch_sizes = df.groupby('batch_id').size()
print(f'Total batches  : {df["batch_id"].nunique()}')
print(f'Rows per batch : min={batch_sizes.min()}  max={batch_sizes.max()}')
print(f'batch_id range : {df["batch_id"].min()} – {df["batch_id"].max()}')
assert batch_sizes.min() == batch_sizes.max() == BATCH_SIZE, 'Unequal batch sizes!'
print('✓ All batches are equal size.')

Total batches  : 90
Rows per batch : min=500  max=500
batch_id range : 0 – 89
✓ All batches are equal size.


## Step 8 — Select Final Columns & Export

Only the columns required by the simulation are retained.
The raw `day` column is dropped in favour of `day_clean` (integer).
The `class` column is kept under its original name, now holding
integer-encoded values (0 = DOWN, 1 = UP).

In [9]:
FINAL_COLUMNS = [
    'date', 'day_clean', 'period',
    'nswprice', 'nswdemand',
    'vicprice', 'vicdemand',
    'transfer',
    'class', 'batch_id'
]

df_clean = df[FINAL_COLUMNS].copy()
print('Final dataset shape:', df_clean.shape)
print()
print('Column summary:')
print(df_clean.dtypes)
print()
df_clean.head()

Final dataset shape: (45000, 10)

Column summary:
date         float64
day_clean      int64
period       float64
nswprice     float64
nswdemand    float64
vicprice     float64
vicdemand    float64
transfer     float64
class          int64
batch_id       int64
dtype: object



,date,day_clean,period,nswprice,nswdemand,vicprice,vicdemand,transfer,class,batch_id
0,0.0,2,0.000000,0.056443,0.439155,0.003467,0.422915,0.414912,1,0
1,0.0,2,0.021277,0.051699,0.415055,0.003467,0.422915,0.414912,1,0
2,0.0,2,0.042553,0.051489,0.385004,0.003467,0.422915,0.414912,1,0
3,0.0,2,0.063830,0.045485,0.314639,0.003467,0.422915,0.414912,1,0
4,0.0,2,0.085106,0.042482,0.251116,0.003467,0.422915,0.414912,0,0


In [10]:
# ── Final class balance check ────────────────────────────────
print('Class distribution in cleaned dataset:')
vc = df_clean['class'].value_counts()
print(f'  DOWN (0): {vc[0]:,}  ({vc[0]/len(df_clean)*100:.1f}%)')
print(f'  UP   (1): {vc[1]:,}  ({vc[1]/len(df_clean)*100:.1f}%)')
print()

# ── Export ──────────────────────────────────────────────────
df_clean.to_excel(OUTPUT_FILE, index=False)
print(f'✓ Saved → {OUTPUT_FILE}')
print(f'  Rows    : {len(df_clean):,}')
print(f'  Batches : {df_clean["batch_id"].nunique()}')
print(f'  Columns : {list(df_clean.columns)}')

Class distribution in cleaned dataset:
  DOWN (0): 25,927  (57.6%)
  UP   (1): 19,073  (42.4%)

✓ Saved → /content/drive/MyDrive/AM's/Self Development/S2 MTI Binus/TESIS-WHEN TO RETRAIN FIX YA/Experiment-Conference Paper/dataset/ready_to_use_datasets/electricity_clean.xlsx
  Rows    : 45,000
  Batches : 90
  Columns : ['date', 'day_clean', 'period', 'nswprice', 'nswdemand', 'vicprice', 'vicdemand', 'transfer', 'class', 'batch_id']
